# 01: Data Exploration

This notebook explores the ClinTox dataset, providing statistics, visualizations, and initial insights into the data.

## Objectives

1. Load ClinTox dataset
2. Report dataset statistics (size, class balance)
3. Visualize example molecules using RDKit
4. Analyze label distributions
5. Prepare data for modeling



In [ ]:
# Setup: Add src to path and import modules
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Import data loading functions
from src.data import load_clintox, load_tox21, get_task_names
from src.utils import set_seed

# Set seed for reproducibility
set_seed(42)

print("✓ Imports successful")



## Load ClinTox Dataset

Load the ClinTox dataset with train/validation/test splits.



In [ ]:
# Load dataset
cache_dir = project_root / "data"
train_df, val_df, test_df = load_clintox(cache_dir=str(cache_dir), split_type="scaffold", seed=42)

print("Dataset loaded successfully!")
print(f"\nTrain set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")
print(f"Total size: {len(train_df) + len(val_df) + len(test_df)}")

# Display first few rows
print("\nFirst few rows of training set:")
print(train_df.head())



## Dataset Statistics

Compute and visualize dataset statistics including class balance and label distributions.



In [ ]:
# Class balance analysis
def analyze_class_balance(df, split_name):
    """Analyze class distribution in a dataset split."""
    total = len(df)
    positive = df['CT_TOX'].sum()
    negative = total - positive
    positive_pct = (positive / total) * 100
    
    print(f"\n{split_name} Set Class Balance:")
    print(f"  Total samples: {total}")
    print(f"  Toxic (1): {positive} ({positive_pct:.2f}%)")
    print(f"  Non-toxic (0): {negative} ({(100-positive_pct):.2f}%)")
    
    return positive, negative, positive_pct

train_pos, train_neg, train_pct = analyze_class_balance(train_df, "Training")
val_pos, val_neg, val_pct = analyze_class_balance(val_df, "Validation")
test_pos, test_neg, test_pct = analyze_class_balance(test_df, "Test")



In [ ]:
# Visualize class balance
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

splits = ['Train', 'Validation', 'Test']
positives = [train_pos, val_pos, test_pos]
negatives = [train_neg, val_neg, test_neg]

for idx, (split, pos, neg) in enumerate(zip(splits, positives, negatives)):
    axes[idx].bar(['Non-toxic (0)', 'Toxic (1)'], [neg, pos], 
                  color=['skyblue', 'salmon'], alpha=0.7)
    axes[idx].set_title(f'{split} Set', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Count')
    axes[idx].grid(axis='y', alpha=0.3)

plt.suptitle('Class Distribution Across Splits', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

# Ensure figures directory exists
if 'figures_dir' not in locals():
    figures_dir = project_root / "output" / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)

# Save figure
fig_path = figures_dir / "01_class_distribution.png"
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"💾 Saved figure: {fig_path.name}")
plt.show()



## Molecule Visualization

Visualize example molecules from the dataset using RDKit.



In [ ]:
# Enhanced molecule visualization with more samples and beautiful styling
from rdkit.Chem import Descriptors
from pathlib import Path

# Create figures directory
figures_dir = project_root / "output" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

def calculate_molecular_properties(mol):
    """Calculate molecular properties for display."""
    if mol is None:
        return None, None, None
    try:
        mw = Descriptors.MolWt(mol)
        num_atoms = mol.GetNumAtoms()
        num_rings = Descriptors.RingCount(mol)
        return mw, num_atoms, num_rings
    except:
        return None, None, None

def visualize_molecules_enhanced(df, n_examples=15, title="Example Molecules", 
                                 class_label=None, mols_per_row=5, save_path=None):
    """Enhanced visualization with molecular properties and better styling."""
    # Sample molecules ensuring diversity
    if len(df) > n_examples:
        sample = df.sample(n_examples, random_state=42)
    else:
        sample = df.copy()
    
    mols = []
    labels = []
    
    for idx, row in sample.iterrows():
        mol = Chem.MolFromSmiles(row['smiles'])
        if mol is not None:
            # Calculate properties
            mw, num_atoms, num_rings = calculate_molecular_properties(mol)
            
            # Create informative label
            toxicity_marker = "☠️ TOXIC" if row['CT_TOX'] == 1 else "✓ Safe"
            if mw is not None:
                label = f"{toxicity_marker}\nMW: {mw:.1f} | Atoms: {num_atoms}"
                if num_rings > 0:
                    label += f" | Rings: {num_rings}"
            else:
                label = toxicity_marker
            
            mols.append(mol)
            labels.append(label)
    
    if len(mols) == 0:
        print(f"No valid molecules to display for {title}")
        return None
    
    # Draw molecules in a grid with enhanced styling
    try:
        img = Draw.MolsToGridImage(
            mols,
            molsPerRow=mols_per_row,
            subImgSize=(350, 350),
            legends=labels,
            maxMols=len(mols)
        )
    except Exception as e:
        print(f"⚠️  Error creating molecule image: {e}")
        return None
    
    # Save figure if path provided
    if save_path and img is not None:
        try:
            save_file_path = figures_dir / save_path
            save_path_str = str(save_file_path)
            
            # RDKit's MolsToGridImage returns a PIL Image
            # Check if save method exists before calling
            if not hasattr(img, 'save'):
                print(f"⚠️  Warning: Image object {type(img)} does not have save method")
                return img
            
            # Save the image
            img.save(save_path_str, format='PNG')
            print(f"💾 Saved figure: {save_path}")
            
        except AttributeError as e:
            print(f"⚠️  AttributeError saving figure {save_path}: {e}")
            print(f"   Image type: {type(img)}, has save: {hasattr(img, 'save')}")
        except Exception as e:
            print(f"⚠️  Warning: Could not save figure {save_path}: {e}")
            import traceback
            traceback.print_exc()
    
    return img

# Display header with statistics
print("=" * 80)
print("🔬 MOLECULE VISUALIZATION - Dataset Overview")
print("=" * 80)
print(f"\n📊 Dataset Statistics:")
print(f"   • Total molecules: {len(train_df)}")
print(f"   • Toxic molecules: {len(train_df[train_df['CT_TOX'] == 1])} ({100*len(train_df[train_df['CT_TOX'] == 1])/len(train_df):.1f}%)")
print(f"   • Non-toxic molecules: {len(train_df[train_df['CT_TOX'] == 0])} ({100*len(train_df[train_df['CT_TOX'] == 0])/len(train_df):.1f}%)")
print("\n" + "=" * 80)

# Visualize TOXIC molecules (15 examples)
print("\n☠️  TOXIC MOLECULES (15 examples)")
print("-" * 80)
toxic_examples = train_df[train_df['CT_TOX'] == 1].sample(min(15, len(train_df[train_df['CT_TOX'] == 1])), random_state=42)
img_toxic = visualize_molecules_enhanced(
    toxic_examples, 
    n_examples=15, 
    title="Toxic Molecules", 
    class_label="Toxic",
    mols_per_row=5,
    save_path="01_toxic_molecules_overview.png"
)
if img_toxic:
    display(img_toxic)



## Diverse Molecule Size Analysis

Visualize molecules across different size ranges to understand structural diversity.


In [ ]:
# Analyze and visualize molecules by size (number of atoms)
from src.featurization import smiles_to_mol

# Ensure figures directory exists
if 'figures_dir' not in locals():
    figures_dir = project_root / "output" / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)

def get_molecule_size(smiles):
    """Get number of atoms in a molecule."""
    mol = smiles_to_mol(smiles)
    if mol is not None:
        return mol.GetNumAtoms()
    return None

# Add molecule sizes to dataframe
train_df['num_atoms'] = train_df['smiles'].apply(get_molecule_size)

# Define size categories
def categorize_size(num_atoms):
    if num_atoms is None:
        return "Unknown"
    elif num_atoms < 15:
        return "Small (<15 atoms)"
    elif num_atoms < 30:
        return "Medium (15-30 atoms)"
    elif num_atoms < 50:
        return "Large (30-50 atoms)"
    else:
        return "Very Large (>50 atoms)"

train_df['size_category'] = train_df['num_atoms'].apply(categorize_size)

# Visualize molecules from different size categories
print("\n" + "=" * 80)
print("📐 MOLECULE SIZE DIVERSITY")
print("=" * 80)

size_categories = ["Small (<15 atoms)", "Medium (15-30 atoms)", "Large (30-50 atoms)", "Very Large (>50 atoms)"]

for size_cat in size_categories:
    category_mols = train_df[train_df['size_category'] == size_cat]
    if len(category_mols) > 0:
        # Get balanced samples from both classes
        toxic_in_cat = category_mols[category_mols['CT_TOX'] == 1]
        nontoxic_in_cat = category_mols[category_mols['CT_TOX'] == 0]
        
        n_per_class = min(5, len(toxic_in_cat), len(nontoxic_in_cat))
        
        if n_per_class > 0:
            samples = pd.concat([
                toxic_in_cat.sample(n_per_class, random_state=42) if len(toxic_in_cat) >= n_per_class else toxic_in_cat,
                nontoxic_in_cat.sample(n_per_class, random_state=42) if len(nontoxic_in_cat) >= n_per_class else nontoxic_in_cat
            ])
            
            print(f"\n🔬 {size_cat}: {len(category_mols)} molecules")
            print(f"   Toxic: {len(toxic_in_cat)}, Non-toxic: {len(nontoxic_in_cat)}")
            
            # Create labels with size info
            mols = []
            labels = []
            for idx, row in samples.iterrows():
                mol = Chem.MolFromSmiles(row['smiles'])
                if mol is not None:
                    mw, num_atoms, num_rings = calculate_molecular_properties(mol)
                    toxicity_marker = "☠️" if row['CT_TOX'] == 1 else "✓"
                    label = f"{toxicity_marker} {num_atoms} atoms\nMW: {mw:.1f}"
                    mols.append(mol)
                    labels.append(label)
            
            if len(mols) > 0:
                img = Draw.MolsToGridImage(
                    mols,
                    molsPerRow=min(5, len(mols)),
                    subImgSize=(300, 300),
                    legends=labels,
                    maxMols=len(mols)
                )
                # Save figure
                try:
                    safe_name = size_cat.replace(' ', '_').replace('(', '').replace(')', '').replace('<', 'lt').replace('>', 'gt').replace('-', '_')
                    fig_path = figures_dir / f"01_molecules_size_{safe_name}.png"
                    img.save(str(fig_path), format='PNG')
                    print(f"💾 Saved figure: {fig_path.name}")
                except AttributeError:
                    try:
                        from PIL import Image as PILImage
                        if hasattr(img, 'tobytes'):
                            pil_img = PILImage.fromarray(img) if hasattr(img, '__array__') else img
                            pil_img.save(str(fig_path), format='PNG')
                            print(f"💾 Saved figure: {fig_path.name}")
                    except Exception as e:
                        print(f"⚠️  Warning: Could not save figure: {e}")
                except Exception as e:
                    print(f"⚠️  Warning: Could not save figure: {e}")
                display(img)


## Summary Statistics Visualization

Create visualizations showing the distribution of molecular properties.


In [ ]:
# Create summary visualizations
# Ensure figures directory exists
if 'figures_dir' not in locals():
    figures_dir = project_root / "output" / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('📊 Molecular Property Distributions', fontsize=16, fontweight='bold', y=0.995)

# 1. Size distribution by class
ax1 = axes[0, 0]
toxic_sizes = train_df[train_df['CT_TOX'] == 1]['num_atoms'].dropna()
nontoxic_sizes = train_df[train_df['CT_TOX'] == 0]['num_atoms'].dropna()

ax1.hist([nontoxic_sizes, toxic_sizes], bins=20, label=['Non-toxic', 'Toxic'], 
         color=['green', 'red'], alpha=0.6, edgecolor='black')
ax1.set_xlabel('Number of Atoms', fontsize=11)
ax1.set_ylabel('Frequency', fontsize=11)
ax1.set_title('Molecular Size Distribution', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Size categories by class
ax2 = axes[0, 1]
size_counts = pd.crosstab(train_df['size_category'], train_df['CT_TOX'])
size_counts.columns = ['Non-toxic', 'Toxic']
size_counts.plot(kind='bar', ax=ax2, color=['green', 'red'], alpha=0.7, edgecolor='black')
ax2.set_xlabel('Size Category', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title('Molecule Count by Size Category', fontsize=12, fontweight='bold')
ax2.legend()
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
ax2.grid(True, alpha=0.3, axis='y')

# 3. Molecular weight distribution
train_df['molecular_weight'] = train_df['smiles'].apply(
    lambda s: Descriptors.MolWt(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) is not None else None
)
toxic_mw = train_df[train_df['CT_TOX'] == 1]['molecular_weight'].dropna()
nontoxic_mw = train_df[train_df['CT_TOX'] == 0]['molecular_weight'].dropna()

ax3 = axes[1, 0]
ax3.hist([nontoxic_mw, toxic_mw], bins=30, label=['Non-toxic', 'Toxic'], 
         color=['green', 'red'], alpha=0.6, edgecolor='black')
ax3.set_xlabel('Molecular Weight (Da)', fontsize=11)
ax3.set_ylabel('Frequency', fontsize=11)
ax3.set_title('Molecular Weight Distribution', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Ring count distribution
train_df['ring_count'] = train_df['smiles'].apply(
    lambda s: Descriptors.RingCount(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) is not None else None
)
toxic_rings = train_df[train_df['CT_TOX'] == 1]['ring_count'].dropna()
nontoxic_rings = train_df[train_df['CT_TOX'] == 0]['ring_count'].dropna()

ax4 = axes[1, 1]
ring_counts = pd.crosstab(train_df['ring_count'].fillna(0).astype(int), train_df['CT_TOX'])
if len(ring_counts.columns) == 2:
    ring_counts.columns = ['Non-toxic', 'Toxic']
    ring_counts.plot(kind='bar', ax=ax4, color=['green', 'red'], alpha=0.7, edgecolor='black', width=0.8)
ax4.set_xlabel('Number of Rings', fontsize=11)
ax4.set_ylabel('Count', fontsize=11)
ax4.set_title('Ring Count Distribution', fontsize=12, fontweight='bold')
ax4.legend()
ax4.set_xticklabels(ax4.get_xticklabels(), rotation=0)
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()

# Save figure
fig_path = figures_dir / "01_molecular_property_distributions.png"
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"💾 Saved figure: {fig_path.name}")
plt.show()

print("\n✅ Enhanced molecule visualization complete!")
# Calculate counts from dataframe instead of referencing variables
n_toxic_displayed = min(15, len(train_df[train_df['CT_TOX'] == 1]))
n_nontoxic_displayed = min(15, len(train_df[train_df['CT_TOX'] == 0]))
print(f"   • Displayed {n_toxic_displayed} toxic molecule examples")
print(f"   • Displayed {n_nontoxic_displayed} non-toxic molecule examples")
if 'size_categories' in locals():
    print(f"   • Analyzed molecules across {len(size_categories)} size categories")


In [ ]:
# Visualize NON-TOXIC molecules (15 examples)
print("\n✓  NON-TOXIC MOLECULES (15 examples)")
print("-" * 80)
nontoxic_examples = train_df[train_df['CT_TOX'] == 0].sample(min(15, len(train_df[train_df['CT_TOX'] == 0])), random_state=42)
img_nontoxic = visualize_molecules_enhanced(
    nontoxic_examples, 
    n_examples=15, 
    title="Non-toxic Molecules", 
    class_label="Non-toxic",
    mols_per_row=5,
    save_path="01_nontoxic_molecules_overview.png"
)
if img_nontoxic:
    display(img_nontoxic)



## Additional Analysis

Let's examine molecule properties and prepare for modeling.



In [ ]:
# Analyze molecular properties
from src.featurization import smiles_to_mol

def analyze_molecular_properties(df, split_name):
    """Analyze basic molecular properties."""
    mols = []
    num_atoms = []
    num_bonds = []
    valid_smiles = []
    
    for smiles in df['smiles']:
        mol = smiles_to_mol(smiles)
        if mol is not None:
            mols.append(mol)
            num_atoms.append(mol.GetNumAtoms())
            num_bonds.append(mol.GetNumBonds())
            valid_smiles.append(smiles)
        else:
            valid_smiles.append(None)
    
    print(f"\n{split_name} Set Molecular Properties:")
    print(f"  Valid molecules: {len(mols)}/{len(df)}")
    if len(mols) > 0:
        print(f"  Avg atoms per molecule: {np.mean(num_atoms):.2f}")
        print(f"  Avg bonds per molecule: {np.mean(num_bonds):.2f}")
        print(f"  Min atoms: {np.min(num_atoms)}, Max atoms: {np.max(num_atoms)}")
    
    return num_atoms, num_bonds

train_atoms, train_bonds = analyze_molecular_properties(train_df, "Training")
val_atoms, val_bonds = analyze_molecular_properties(val_df, "Validation")
test_atoms, test_bonds = analyze_molecular_properties(test_df, "Test")



In [ ]:
# Visualize molecular size distribution
# Ensure figures directory exists
if 'figures_dir' not in locals():
    figures_dir = project_root / "output" / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_atoms, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Number of Atoms')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Atoms per Molecule (Training Set)')
axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(train_bonds, bins=30, alpha=0.7, color='salmon', edgecolor='black')
axes[1].set_xlabel('Number of Bonds')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Bonds per Molecule (Training Set)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()

# Save figure
fig_path = figures_dir / "01_atom_bond_distribution.png"
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"💾 Saved figure: {fig_path.name}")
plt.show()



## Summary

✓ Dataset loaded successfully  
✓ Class balance analyzed  
✓ Example molecules visualized  
✓ Molecular properties examined  

**Next Steps:**
1. Proceed to `02_training_baseline.ipynb` for fingerprint-based baseline model
2. Or proceed to `03_training_gnn.ipynb` for torch-molecule GNN model

